In [ ]:
# Notebook formatting
%load_ext jupyter_black

# Visualisations - Multinational Corporation Profit Shiftings

This notebook produces charts and figures to analyse the evolution of  Multinational corporation (MNC) profit shiftings

## Setup

### Imports

In [ ]:
# Data wrangling
import pandas as pd
import tjn_tools

# Visualisations
import plotly.graph_objs as go
import plotly.io as pio

# Configuration
from config import YEARS

### Global Variables and Constants

In [ ]:
# Dictionnary of filenames for outward profit shifting (PSO) data, indexed by year
FILENAMES_PSO_DICT = {year: f"profits_shifting_outward_{year}.xlsx" for year in YEARS}

# Paths
PSO_PATH = "../../data/intermediate/analysis/"
FINAL_DATA_FOLDER_PATH = "../../data/final/analysis/"
FINAL_FIGURES_FOLDER_PATH = "../..data/final/figures/"

## Main

### Data Wrangling

In [ ]:
def retrieve_profit_shifting_outward(filenames: dict):
    """Retrieve profit shifting outward data from intermediate folder, merge them in a single dataset and save it in
    the intermediate folder.
    """

    def retrieve_dataset(year):
        """Retrieve and preprocess single dataset from intermediate folder"""
        df = pd.read_excel(PSO_PATH + filenames[year])
        df.rename(
            columns={"shifted_profits_outward": f"profits_shifting_outward_{year}"},
            inplace=True,
        )
        df = df.loc[:, ["iso3", f"profits_shifting_outward_{year}"]]
        return df

    # Merge all datasets
    output = retrieve_dataset(YEARS[0])
    for year in YEARS[1:]:
        df = retrieve_dataset(year)
        output = pd.merge(output, df, on="iso3", how="outer", validate="1:1")

    # Add country names
    output.insert(1, "name", output["iso3"].apply(tjn_tools.iso3_to_name))

    return output


profits_shifting = retrieve_profit_shifting_outward(FILENAMES_PSO_DICT)
profits_shifting

### Visualisations

#### Line Charts

In [ ]:
def create_line_chart(
    x_values, y_values, title, x_axis_title, y_axis_title, filename, path
):
    """Create a line chart and save it as a PNG image"""
    # Create a trace for the line chart
    trace = go.Scatter(x=x_values, y=y_values, mode="lines")

    # Create the layout for the chart
    layout = go.Layout(
        title={
            "text": title,
            "x": 0.5,
            "y": 0.9,
            "xanchor": "center",
            "yanchor": "top",
        },
        xaxis=dict(title=x_axis_title),
        yaxis=dict(title=y_axis_title),
    )

    # Combine the trace and layout into a figure
    fig = go.Figure(data=[trace], layout=layout)

    # Save the chart as a PNG image
    pio.write_image(
        fig,
        path + filename,
    )

In [ ]:
# Create a line chart for each country displaying the evolution of profit shifting outward over the years.
def create_line_chart_for_each_country(df):
    # Iterate over each row of the profits_shifting dataset
    for index, row in profits_shifting.iterrows():
        x_values = [str(year) for year in YEARS]
        y_values = row[~row.index.isin(["iso3", "name"])].tolist()
        title = f"Evolution of Profit Shifting Outward in {row['name']}"
        x_axis_title = "Year"
        y_axis_title = "Profit Shifting Outward (USD)"
        filename = f"{row['iso3']}_evolution_pso.png"
        path = "../../data/final/figures/"

        create_line_chart(
            x_values, y_values, title, x_axis_title, y_axis_title, filename, path
        )


create_line_chart_for_each_country(profits_shifting)

#### Rankings

In [ ]:
# Create function to rank countries by their absolute increase in profit shifting outward between YEARS
def rank_countries_by_pso_absolute_increase(df: pd.DataFrame) -> pd.DataFrame:
    """Rank countries by their increase in profit shifting outward between the first and last year of the dataset"""
    first_year, last_year = YEARS[0], YEARS[-1]

    # Calculate the absolute increase in profit shifting outward between the years
    profit_increase = (
        df[f"profits_shifting_outward_{last_year}"]
        - df[f"profits_shifting_outward_{first_year}"]
    )

    # Create a new DataFrame with the absolute increase and country name
    rank_data = pd.DataFrame(
        {
            "iso3": df["iso3"],
            "country": df["name"],
            f"profits_shifting_outward_{first_year}": df[
                f"profits_shifting_outward_{first_year}"
            ],
            f"profits_shifting_outward_{last_year}": df[
                f"profits_shifting_outward_{last_year}"
            ],
            "pso_abs_increase": profit_increase,
        }
    )

    # Sort the DataFrame by the absolute increase in profit shifting outward
    rank_data.sort_values(by="pso_abs_increase", ascending=False, inplace=True)

    # Reset the index
    rank_data.reset_index(drop=True, inplace=True)

    # Save the DataFrame as an excel file
    rank_data.to_excel(
        FINAL_DATA_FOLDER_PATH + "ranking_profit_shiftings_absolute_increase.xlsx",
        index=False,
    )

    return rank_data


ranking_pso_abs_increase = rank_countries_by_pso_absolute_increase(profits_shifting)
ranking_pso_abs_increase

In [ ]:
# Create function to rank countries by their increase rate in profit shifting outward between YEARS
def rank_countries_by_pso_rate_increase(df: pd.DataFrame) -> pd.DataFrame:
    """Rank countries by their rate increase in profit shifting outward between the first and last year of the dataset"""
    first_year, last_year = YEARS[0], YEARS[-1]

    # Calculate the absolute increase in profit shifting outward between the years
    profit_rel_increase = (
        df[f"profits_shifting_outward_{last_year}"]
        / df[f"profits_shifting_outward_{first_year}"]
    )

    # Create a new DataFrame with the absolute increase and country name
    rank_data = pd.DataFrame(
        {
            "iso3": df["iso3"],
            "country": df["name"],
            f"profits_shifting_outward_{first_year}": df[
                f"profits_shifting_outward_{first_year}"
            ],
            f"profits_shifting_outward_{last_year}": df[
                f"profits_shifting_outward_{last_year}"
            ],
            "pso_rate_increase": profit_rel_increase,
        }
    )

    # Sort the DataFrame by the absolute increase in profit shifting outward
    rank_data.sort_values(by="pso_rate_increase", ascending=False, inplace=True)

    # Reset the index
    rank_data.reset_index(drop=True, inplace=True)

    # Save the DataFrame as an excel file
    rank_data.to_excel(
        FINAL_DATA_FOLDER_PATH + "ranking_profit_shiftings_rate_increase.xlsx",
        index=False,
    )

    return rank_data


ranking_pso_rel_increase = rank_countries_by_pso_rate_increase(profits_shifting)
ranking_pso_rel_increase